# **PREPROCESSING OF DATASET: SUBTITLES**

## Table of Contents
1. [Setup](#Setup)
2. [Step 1: Subtitle Cleaning](#step-1-subtitle-cleaning)
3. [Step 2: Sentence Splitting](#Step-2-Sentence-Splitting)
4. [Step 3: Tokenization for Topic Modeling](#Step-3-Tokenization-for-Topic-Modeling)
5. [References](#references)

## Setup

This first section installs and loads all the necessary libraries for our subtitle preprocessing. We use 'nltk' for text processing, regular expressions for cleaning, and OS tools to handle file operations.


In [1]:
# Install if needed
%pip install nltk
%pip install regex

import os
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer

# Download NLTK resources
nltk.download('stopwords')


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## STEP 1: Subtitle Cleaning

Defining the cleaning function. It reads all .srt subtitle files from the designated folder and removes unwanted elements such as timestamps, speaker names, HTML tags, and non-verbal sound cues like '[LAUGHING]' or '♪', for example. The result is a cleaner version of the subtitles saved in .txt files, which we will futher process later.

In [2]:

def clean_srt_line(line):
    # remove invisible characters
    line = re.sub(r'[\u200b\uFEFF\u202a\u202c\u202d\u200e\u200f]', '', line).strip()

    # remove dashes at beginning of sentence
    line = re.sub(r'^\s*-+', '', line).strip()

    # remove HTML tags like <i> and </i>
    line = re.sub(r'</?i>', '', line)

    # remove music cues and sound directions
    line = re.sub(r'\[.*?\]', '', line)    # for example: [music], [gasp]
    line = line.replace('♪', '')           # music note character

    # remove all-caps words inside parentheses or brackets
    line = re.sub(r'\((?:[A-Z\s]{2,})\)', '', line)  # (LAUGHING)
    line = re.sub(r'\[(?:[A-Z\s]{2,})\]', '', line)  # [MUSIC]

    # remove speaker names in all caps followed by a colon (for example: "MIGUEL:")
    line = re.sub(r'^[A-Z\s]+:', '', line).strip()

    # skip block numbers
    if re.match(r'^\d+$', line):
        return None

    # skip timestamp lines
    if re.match(r'\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}', line):
        return None

    # skip empty lines
    if line == '':
        return None

    return line

**Cleaning for Disney and Ghibli**

Here we apply the cleaning function to both the Disney and Ghibli subtitle folders. Each movie will result in a new cleaned text file stored in 'data_cleaned/disney' or 'data_cleaned/ghibli'.

In [3]:
def clean_all_srt_files(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for file in os.listdir(input_dir):
        if os.path.isfile(os.path.join(input_dir, file)):
            input_path = os.path.join(input_dir, file)
            base_name = os.path.splitext(file)[0]
            output_filename = base_name + '_cleaned.txt'
            output_path = os.path.join(output_dir, output_filename)

            with open(input_path, 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.readlines()

            cleaned_lines = [clean_srt_line(line) for line in lines if clean_srt_line(line)]

            with open(output_path, 'w', encoding='utf-8') as f:
                for line in cleaned_lines:
                    f.write(line + '\n')

            print(f"Cleaned: {file} → {output_filename}")

In [5]:
clean_all_srt_files('data/initial_dataset/disney_initial_dataset', 'data/data_cleaned/disney')
clean_all_srt_files('data/initial_dataset/ghibli_initial_dataset', 'data/data_cleaned/ghibli')

Cleaned: Ralph.Breaks.the.Internet.srt → Ralph.Breaks.the.Internet_cleaned.txt
Cleaned: Toy.Story.srt → Toy.Story_cleaned.txt
Cleaned: Brave.srt → Brave_cleaned.txt
Cleaned: Dumbo.srt → Dumbo_cleaned.txt
Cleaned: Beauty.and.the.Beast.srt → Beauty.and.the.Beast_cleaned.txt
Cleaned: Wreck-It.Ralph.srt → Wreck-It.Ralph_cleaned.txt
Cleaned: Pinocchio.srt → Pinocchio_cleaned.txt
Cleaned: Pocahontas.srt → Pocahontas_cleaned.txt
Cleaned: Moana.srt → Moana_cleaned.txt
Cleaned: Hercules.srt → Hercules_cleaned.txt
Cleaned: Frozen.srt → Frozen_cleaned.txt
Cleaned: The.Little.Mermaid.srt → The.Little.Mermaid_cleaned.txt
Cleaned: Cinderella.srt → Cinderella_cleaned.txt
Cleaned: Sleeping.Beauty.srt → Sleeping.Beauty_cleaned.txt
Cleaned: Snow.White.and.the.Seven.Dwarfs.srt → Snow.White.and.the.Seven.Dwarfs_cleaned.txt
Cleaned: Tangled.srt → Tangled_cleaned.txt
Cleaned: Chicken.Little.srt → Chicken.Little_cleaned.txt
Cleaned: The.Lion.King.srt → The.Lion.King_cleaned.txt
Cleaned: Zootopia.srt → Zootop

## STEP 2: Sentence Splitting

For sentiment analysis, we need to break the subtitle text into individual sentences. This function removes non-verbal markers and music cues, then uses punctuation to detect where sentences end. It outputs one sentence per line per movie.


In [6]:
def simple_sentence_split(text):
    # split on punctuation that ends a sentence, followed by a space
    return re.split(r'(?<=[.!?])\s+', text.strip())

**Generate Sentence Files**

This runs the sentence splitting function for both studios. It saves each movie’s sentences in a separate file, and also creates a combined file with all sentences from Disney or Ghibli, which will be used for sentence-level sentiment analysis.


In [7]:
def split_subtitles_into_sentences(input_folder, output_folder, combined_output_file):
    os.makedirs(output_folder, exist_ok=True)
    all_sentences = []

    for filename in os.listdir(input_folder):
        input_path = os.path.join(input_folder, filename)

        if os.path.isfile(input_path):
            with open(input_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            # join all cleaned lines (one big text block)
            text = ' '.join(line.strip() for line in lines if line.strip())
            sentences = simple_sentence_split(text)
            all_sentences.extend(sentences)

            # save sentence file for each movie
            base_name = os.path.splitext(filename)[0]
            output_path = os.path.join(output_folder, base_name + '_sentences.txt')
            with open(output_path, 'w', encoding='utf-8') as f_out:
                for sentence in sentences:
                    f_out.write(sentence.strip() + '\n')

            print(f"Split {filename} → {output_path}")

    # save one file with all the sentences for the same studio
    with open(combined_output_file, 'w', encoding='utf-8') as f_combined:
        for sentence in all_sentences:
            f_combined.write(sentence.strip() + '\n')

    print(f"\nCombined sentences saved to: {combined_output_file}")


In [8]:
split_subtitles_into_sentences(
    input_folder='data/data_cleaned/disney',
    output_folder='data/data_in_sentences/disney',
    combined_output_file='data/data_in_sentences/disney_sentences.txt'
)

split_subtitles_into_sentences(
    input_folder='data/data_cleaned/ghibli',
    output_folder='data/data_in_sentences/ghibli',
    combined_output_file='data/data_in_sentences/ghibli_sentences.txt'
)


Split Sleeping.Beauty_cleaned.txt → data/data_in_sentences/disney/Sleeping.Beauty_cleaned_sentences.txt
Split Pocahontas_cleaned.txt → data/data_in_sentences/disney/Pocahontas_cleaned_sentences.txt
Split Brave_cleaned.txt → data/data_in_sentences/disney/Brave_cleaned_sentences.txt
Split Moana_cleaned.txt → data/data_in_sentences/disney/Moana_cleaned_sentences.txt
Split Beauty.and.the.Beast_cleaned.txt → data/data_in_sentences/disney/Beauty.and.the.Beast_cleaned_sentences.txt
Split The.Little.Mermaid_cleaned.txt → data/data_in_sentences/disney/The.Little.Mermaid_cleaned_sentences.txt
Split Wreck-It.Ralph_cleaned.txt → data/data_in_sentences/disney/Wreck-It.Ralph_cleaned_sentences.txt
Split Tangled_cleaned.txt → data/data_in_sentences/disney/Tangled_cleaned_sentences.txt
Split Snow.White.and.the.Seven.Dwarfs_cleaned.txt → data/data_in_sentences/disney/Snow.White.and.the.Seven.Dwarfs_cleaned_sentences.txt
Split Lilo.&.Stitch_cleaned.txt → data/data_in_sentences/disney/Lilo.&.Stitch_cleane

## STEP 3: Tokenization for Topic Modeling

To prepare the text for topic modeling, we break it into individual words (tokens), remove stopwords (common words like "the" or "and"), and convert everything to lowercase. This helps us focus on the most meaningful words in the subtitles.

In [9]:
def tokenize_text(text):
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(text)
    stop_words = set(stopwords.words('english'))

    cleaned_tokens = [
        word.lower() for word in tokens
        if word.lower() not in stop_words
    ]
    return cleaned_tokens


**Tokenize All Movies**

This function tokenizes the cleaned subtitle text of each movie and saves the result as a single space-separated string of words. These files are stored in 'data/data_tokenized/disney' and 'data/data_tokenized/ghibli'. Later, we merge all token files for each studio into one master file. This final file will be used for LDA topic modeling to extract overall themes in the movies.


In [10]:
def preprocess_text(text):
    # Use RegexpTokenizer to remove punctuation
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(text)
    stop_words = set(stopwords.words('english'))

    # Remove stopwords and lowercase all words
    cleaned_tokens = [
        word.lower() for word in tokens
        if word.lower() not in stop_words
    ]
    return cleaned_tokens

def tokenize_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):
        input_path = os.path.join(input_folder, file)
        if os.path.isfile(input_path) and file.endswith('.txt'):
            with open(input_path, 'r', encoding='utf-8') as f:
                text = f.read()

            tokens = preprocess_text(text)

            base_name = os.path.splitext(file)[0]
            output_file = os.path.join(output_folder, base_name + '_tokens.txt')
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(' '.join(tokens))

            print(f"Tokenized: {file} → {base_name + '_tokens.txt'}")

In [11]:
def combine_tokenized_files(input_folder, output_path):
    all_tokens = []
    for file in os.listdir(input_folder):
        file_path = os.path.join(input_folder, file)
        if os.path.isfile(file_path) and file.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                tokens = f.read().split()
                all_tokens.extend(tokens)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(' '.join(all_tokens))

    print(f"Combined {len(all_tokens)} tokens into: {output_path}")


In [12]:
tokenize_folder('data/data_cleaned/disney', 'data/data_preprocessed/disney')
tokenize_folder('data/data_cleaned/ghibli', 'data/data_preprocessed/ghibli')

combine_tokenized_files('data/data_tokenized/disney', 'data/data_tokenized/disney_tokenized.txt')
combine_tokenized_files('data/data_tokenized/ghibli', 'data/data_tokenized/ghibli_tokenized.txt')

Tokenized: Sleeping.Beauty_cleaned.txt → Sleeping.Beauty_cleaned_tokens.txt
Tokenized: Pocahontas_cleaned.txt → Pocahontas_cleaned_tokens.txt
Tokenized: Brave_cleaned.txt → Brave_cleaned_tokens.txt
Tokenized: Moana_cleaned.txt → Moana_cleaned_tokens.txt
Tokenized: Beauty.and.the.Beast_cleaned.txt → Beauty.and.the.Beast_cleaned_tokens.txt
Tokenized: The.Little.Mermaid_cleaned.txt → The.Little.Mermaid_cleaned_tokens.txt
Tokenized: Wreck-It.Ralph_cleaned.txt → Wreck-It.Ralph_cleaned_tokens.txt
Tokenized: Tangled_cleaned.txt → Tangled_cleaned_tokens.txt
Tokenized: Snow.White.and.the.Seven.Dwarfs_cleaned.txt → Snow.White.and.the.Seven.Dwarfs_cleaned_tokens.txt
Tokenized: Lilo.&.Stitch_cleaned.txt → Lilo.&.Stitch_cleaned_tokens.txt
Tokenized: Dumbo_cleaned.txt → Dumbo_cleaned_tokens.txt
Tokenized: Frozen_cleaned.txt → Frozen_cleaned_tokens.txt
Tokenized: Chicken.Little_cleaned.txt → Chicken.Little_cleaned_tokens.txt
Tokenized: Zootopia_cleaned.txt → Zootopia_cleaned_tokens.txt
Tokenized: Cin

## References

The initial text cleaning phase was inspired by best practices outlined for efficient text preprocessing in Python (GeeksforGeeks, 2021). This included removing invisible Unicode characters, redundant whitespace, formatting tags such as HTML tags, speaker annotations in all caps, and stage directions in brackets or parentheses. These steps help reduce noise in subtitle data and improve the accuracy of sentence segmentation and tokenization.

Sentence segmentation was performed using a regular expression-based approach, aligned with standard techniques described by Bird et al. (2009) for handling informal or unstructured text such as subtitles.

Tokenization and stopword removal were implemented using NLTK’s RegexpTokenizer and its built-in English stopword list (Bird et al., 2009), ensuring that uninformative words (e.g., the, is, in) were excluded before topic modeling.

In addition, the Jupyter notebooks and code examples provided in the AUC Text Mining GitHub repository were used as a reference throughout the preprocessing steps.

**Citations**

- Bird, S., Klein, E., & Loper, E. (2009). *Natural Language Processing with Python: Analyzing Text with the Natural Language Toolkit.* O'Reilly Media.

- Bloem, J. (2025). *AUC Text Mining Course Resources.* GitHub Repository. Retrieved from https://github.com/bloemj/AUC_TM_2025

- GeeksforGeeks. (2021). *Python – Efficient Text Data Cleaning.* Retrieved from https://www.geeksforgeeks.org/python-efficient-text-data-cleaning/